In [1]:

!pip -q install gradio scikit-learn pandas numpy


In [2]:

import numpy as np
import pandas as pd

np.random.seed(42)
n = 1500

df = pd.DataFrame({
    "attendance": np.random.uniform(40, 100, n),
    "assignment_avg": np.random.uniform(20, 100, n),
    "midterm_score": np.random.uniform(10, 100, n),
    "study_hours": np.random.uniform(0, 35, n),
    "previous_score": np.random.uniform(20, 100, n),
    "sleep_hours": np.random.uniform(4, 10, n)
})

noise = np.random.normal(0, 5, n)

df["final_score"] = np.clip(
    0.20 * df["attendance"]
    + 0.20 * df["assignment_avg"]
    + 0.30 * df["midterm_score"]
    + 0.80 * df["study_hours"]
    + 0.20 * df["previous_score"]
    + 1.0 * df["sleep_hours"]
    - 5
    + noise,
    0, 100
)

print("Dataset shape:", df.shape)
display(df.head(10))


Dataset shape: (1500, 7)


,attendance,assignment_avg,midterm_score,study_hours,previous_score,sleep_hours,final_score
0,62.472407,61.526543,70.543269,20.975475,71.860556,7.257269,79.637827
1,97.042858,58.334550,81.701326,17.979775,33.790909,8.997362,85.051692
2,83.919637,22.051365,32.542111,10.086458,89.791565,4.750533,54.350465
3,75.919509,47.299826,66.238669,0.226253,69.049299,4.778698,66.670607
4,49.361118,50.415650,61.457138,17.368373,32.576311,7.340910,53.840861
5,49.359671,51.905822,84.954734,9.996910,96.987045,9.678272,83.769031
6,43.485017,66.413790,91.547835,25.711291,61.469237,6.489987,87.172664
7,91.970569,62.688204,11.094109,0.838119,25.831876,5.961653,32.478344
8,76.066901,68.632407,70.661793,20.468973,70.146632,7.640136,96.737012
9,82.484355,81.190661,14.665222,32.951598,40.255913,9.582692,75.810996


In [3]:

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

features = [
    "attendance",
    "assignment_avg",
    "midterm_score",
    "study_hours",
    "previous_score",
    "sleep_hours"
]

X = df[features]
y = df["final_score"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

model = RandomForestRegressor(
    n_estimators=150,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

print("Training completed!")
print("Mean Absolute Error:", round(mean_absolute_error(y_test, predictions), 2))
print("R² score:", round(r2_score(y_test, predictions), 3))

importance = pd.Series(
    model.feature_importances_,
    index=features
).sort_values(ascending=False)

display(importance.to_frame("Feature importance"))


Training completed!
Mean Absolute Error: 4.79
R² score: 0.82


,Feature importance
study_hours,0.325443
midterm_score,0.316353
previous_score,0.127538
assignment_avg,0.117761
attendance,0.081341
sleep_hours,0.031564


In [4]:

def generate_recommendations(attendance, assignment_avg,
                             midterm_score, study_hours,
                             previous_score, sleep_hours,
                             predicted_score):

    tips = []

    if attendance < 75:
        tips.append(
            "Attendance is below 75%. Attend classes regularly "
            "and check your institution's attendance requirements."
        )

    if assignment_avg < 60:
        tips.append(
            "Improve assignment performance. Practise questions "
            "and submit work before deadlines."
        )

    if midterm_score < 50:
        tips.append(
            "Revise midterm topics and solve previous exam questions."
        )

    if study_hours < 10:
        tips.append(
            "Try a consistent study timetable with focused daily sessions."
        )

    if previous_score < 50:
        tips.append(
            "Strengthen fundamentals before moving to advanced topics."
        )

    if sleep_hours < 7:
        tips.append(
            "Consider maintaining a regular, adequate sleep schedule."
        )

    if predicted_score < 40:
        tips.append(
            "Prioritize foundational concepts and ask a teacher for support."
        )
    elif predicted_score < 70:
        tips.append(
            "Use weekly revision, practice tests, and error review."
        )
    else:
        tips.append(
            "Maintain your routine and practise timed mock examinations."
        )

    if not tips:
        tips.append(
            "Keep monitoring your progress and maintain your study routine."
        )

    return tips


In [ ]:

import gradio as gr

def predict_student(attendance, assignment_avg, midterm_score,
                    study_hours, previous_score, sleep_hours):

    values = pd.DataFrame([{
        "attendance": attendance,
        "assignment_avg": assignment_avg,
        "midterm_score": midterm_score,
        "study_hours": study_hours,
        "previous_score": previous_score,
        "sleep_hours": sleep_hours
    }])[features]

    score = float(np.clip(model.predict(values)[0], 0, 100))

    if score < 40:
        category = "Needs additional support"
    elif score < 60:
        category = "Developing"
    elif score < 75:
        category = "Good progress"
    else:
        category = "Strong progress"

    tips = generate_recommendations(
        attendance, assignment_avg, midterm_score,
        study_hours, previous_score, sleep_hours, score
    )

    recommendations = "\n".join(
        f"{i}. {tip}" for i, tip in enumerate(tips, 1)
    )

    return (
        f"**Predicted final score: {score:.1f}/100**\n\n"
        f"**Performance category:** {category}",
        recommendations
    )

with gr.Blocks(title="Student Success AI") as student_app:
    gr.Markdown("# 🎓 Student Success AI")
    gr.Markdown(
        "Enter student details to estimate a final score "
        "and receive personalized study suggestions."
    )

    with gr.Row():
        attendance = gr.Slider(0, 100, value=80, label="Attendance (%)")
        assignment = gr.Slider(0, 100, value=75, label="Assignment average (%)")

    with gr.Row():
        midterm = gr.Slider(0, 100, value=65, label="Midterm score (%)")
        study = gr.Slider(0, 35, value=12, label="Study hours per week")

    with gr.Row():
        previous = gr.Slider(0, 100, value=70, label="Previous score (%)")
        sleep = gr.Slider(4, 10, value=7.5, label="Sleep hours per night")

    run_button = gr.Button("Predict score", variant="primary")
    result = gr.Markdown()
    recommendations = gr.Markdown()

    run_button.click(
        fn=predict_student,
        inputs=[
            attendance, assignment, midterm,
            study, previous, sleep
        ],
        outputs=[result, recommendations]
    )

student_app.launch(share=True, debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://0eead392c190efbe5a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
